# **Install Gemini SDK and Configure API Authentication**

In [2]:
!pip install -q google-genai

from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# **Test the Gemini API Connection**

In [4]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello in one short sentence."
)
print(response.text)

Hello! How can I help you today?


# **Load the Anomaly Detection Results**

In [5]:
import pandas as pd

df = pd.read_csv("/content/anomaly_results.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

,timestamp,value,rolling_avg_24h,rolling_std_24h,lag_1h,diff_1h,anomaly
0,2013-07-04 23:00:00,70.649957,70.470846,1.012776,72.187695,-1.537738,1
1,2013-07-05 00:00:00,71.342742,70.531759,1.019686,70.649957,0.692785,1
2,2013-07-05 01:00:00,71.586728,70.547030,1.033101,71.342742,0.243986,1
3,2013-07-05 02:00:00,70.977001,70.551163,1.034679,71.586728,-0.609727,1
4,2013-07-05 03:00:00,70.243882,70.604683,0.980569,70.977001,-0.733119,1


# **Extract One Anomaly to Test the Explanation**

In [6]:
sample_anomaly = df[df['anomaly'] == -1].iloc[0]
print(sample_anomaly)

timestamp          2013-12-21 20:00:00
value                         82.28924
rolling_avg_24h              78.603466
rolling_std_24h               1.206952
lag_1h                       79.896875
diff_1h                       2.392365
anomaly                             -1
Name: 3674, dtype: object


# **Build the Explanation Prompt**

In [7]:
prompt = f"""
You are explaining an anomaly detected in ambient temperature sensor data to a non-technical operations manager.

Anomaly details:
- Timestamp: {sample_anomaly['timestamp']}
- Actual temperature reading: {sample_anomaly['value']:.2f}
- 24-hour rolling average at this time: {sample_anomaly['rolling_avg_24h']:.2f}
- Change from the previous hour: {sample_anomaly['diff_1h']:.2f}

Write a short, clear explanation (2-3 sentences) of what this anomaly likely means and why it was flagged as unusual. Avoid technical jargon.
"""

print(prompt)


You are explaining an anomaly detected in ambient temperature sensor data to a non-technical operations manager.

Anomaly details:
- Timestamp: 2013-12-21 20:00:00
- Actual temperature reading: 82.29
- 24-hour rolling average at this time: 78.60
- Change from the previous hour: 2.39

Write a short, clear explanation (2-3 sentences) of what this anomaly likely means and why it was flagged as unusual. Avoid technical jargon.



# **Send the Prompt to Gemini and Get the Explanation**

In [8]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)
print(response.text)

At 8:00 PM on December 21, the temperature rapidly spiked to 82.3°F, rising nearly 2.4 degrees in a single hour to sit well above the day's average of 78.6°F. The system flagged this because temperatures rarely jump this quickly on a winter evening, when room conditions should be stable or cooling down. This sudden rise likely points to a heating system malfunction, an unmonitored heat source near the sensor, or an external vent or door left open.


# **Automate for All Detected Anomalies**

In [9]:
def explain_anomaly(row):
    prompt = f"""
You are explaining an anomaly detected in ambient temperature sensor data to a non-technical operations manager.

Anomaly details:
- Timestamp: {row['timestamp']}
- Actual temperature reading: {row['value']:.2f}
- 24-hour rolling average at this time: {row['rolling_avg_24h']:.2f}
- Change from the previous hour: {row['diff_1h']:.2f}

Write a short, clear explanation (2-3 sentences) of what this anomaly likely means and why it was flagged as unusual. Avoid technical jargon.
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

# **Test the Function on Another Anomaly**

In [10]:
another_anomaly = df[df['anomaly'] == -1].iloc[10]
explanation = explain_anomaly(another_anomaly)
print(explanation)

At 6:00 AM on December 22, the system flagged a reading of 82.1° because the area remained unexpectedly warm compared to its 24-hour average of 80.4°. Even though the temperature dropped slightly from the previous hour, maintaining such a high reading during the early morning—when the space is typically at its coolest—is unusual. This likely points to an issue with the overnight cooling system or an unexpected heat source remaining active.


# **Generate Explanations for All Anomalies and Save Results**

In [11]:
sample_batch = df[df['anomaly'] == -1].head(5).copy()
sample_batch['explanation'] = sample_batch.apply(explain_anomaly, axis=1)
sample_batch[['timestamp', 'value', 'explanation']]

,timestamp,value,explanation
3674,2013-12-21 20:00:00,82.289240,"At 8:00 PM on December 21st, the sensor record..."
3675,2013-12-21 21:00:00,82.989869,"At 9:00 PM on December 21st, the ambient tempe..."
3676,2013-12-21 22:00:00,83.247886,"On December 21 at 10:00 PM, the sensor recorde..."
3677,2013-12-21 23:00:00,82.519659,"At 11:00 PM on December 21, the sensor recorde..."
3678,2013-12-22 00:00:00,82.736802,"At midnight on December 22nd, the space reache..."


# **Generate Explanations for All Detected Anomalies**

In [13]:
dec_sample = df[(df['timestamp'] >= '2013-12-21') & (df['timestamp'] <= '2013-12-23') & (df['anomaly'] == -1)].head(10)
jan_sample = df[(df['timestamp'] >= '2014-01-12') & (df['timestamp'] <= '2014-01-13') & (df['anomaly'] == -1)]
other_sample = df[df['anomaly'] == -1].sample(10, random_state=42)

selected_anomalies = pd.concat([dec_sample, jan_sample, other_sample]).drop_duplicates().copy()
print(f"Total selected for explanation: {len(selected_anomalies)}")

Total selected for explanation: 22


# **Combine All Successful Explanations**

In [15]:
final_results = sample_batch[['timestamp', 'value', 'rolling_avg_24h', 'diff_1h', 'explanation']].copy()
final_results.to_csv("anomaly_explanations.csv", index=False)
final_results

,timestamp,value,rolling_avg_24h,diff_1h,explanation
3674,2013-12-21 20:00:00,82.289240,78.603466,2.392365,"At 8:00 PM on December 21st, the sensor record..."
3675,2013-12-21 21:00:00,82.989869,78.851674,0.700629,"At 9:00 PM on December 21st, the ambient tempe..."
3676,2013-12-21 22:00:00,83.247886,79.088685,0.258017,"On December 21 at 10:00 PM, the sensor recorde..."
3677,2013-12-21 23:00:00,82.519659,79.231509,-0.728227,"At 11:00 PM on December 21, the sensor recorde..."
3678,2013-12-22 00:00:00,82.736802,79.407181,0.217143,"At midnight on December 22nd, the space reache..."
